# Robo Flow — Stock Check (Google Colab GPU)
1. Cell 2 run karein aur app se copied config URL paste karein.
2. Bas isi code cell ko Run karein — install aur Stock Check automatically start hoga.
3. Progress aur results Stock Check page par nazar aayenge. DB mein images/results save nahi honge.


In [ ]:
import json, os, subprocess, sys, urllib.request
CONFIG_URL = input('Paste Stock Check config URL, then Enter: ').strip()
if not CONFIG_URL: raise SystemExit('Config URL missing — app mein Open Colab & Check dobara click karein')
with urllib.request.urlopen(CONFIG_URL, timeout=60) as response:
    cfg = json.loads(response.read().decode('utf-8'))
for key, cfg_key in [('SUPABASE_URL','supabase_url'),('SUPABASE_SERVICE_ROLE_KEY','supabase_service_role_key'),('HF_TOKEN','hf_token'),('HF_DATASET_REPO','hf_dataset_repo'),('HF_MODEL_REPO','hf_model_repo'),('WORKER_API_KEY','worker_api_key')]:
    os.environ[key] = cfg.get(cfg_key) or ''
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['DEPLOY_TARGET'] = 'colab'
os.environ['LOW_MEMORY_MODE'] = 'false'
os.environ['MEMORY_SOFT_LIMIT_MB'] = '10000'
os.environ['MEMORY_HARD_LIMIT_MB'] = '12000'
os.environ['INFERENCE_MAX_IMAGE_SIZE'] = '640'
os.environ['INFERENCE_MIN_IMAGE_SIZE'] = '416'
os.environ['YOLO_IMGSZ'] = '640'
print('Config loaded:', len(cfg['image_urls']), 'Result Image URLs')
REPO_URL = cfg.get('repo_url', 'https://github.com/adeeltariq6480/robo_flow.git')
WORKDIR = '/content/robo_flow'
if not os.path.isdir(WORKDIR + '/.git'):
    subprocess.run(['git', 'clone', REPO_URL, WORKDIR], check=True)
else:
    subprocess.run(['git', '-C', WORKDIR, 'pull'], check=False)
os.chdir(WORKDIR + '/worker')
print('Installing packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Starting Stock Check on Colab GPU...')
result = subprocess.run([sys.executable, 'scripts/colab_stock_url_check.py', '--config-url', CONFIG_URL], check=False)
if result.returncode: raise SystemExit(result.returncode)
print('Stock Check complete — results app mein dekhein.')
